# Number of AP and IB courses per school (2017-18)

- **Paper:** "Number of AP Classes" variable (Table 1)
- **Original notebook:** `College_courses.ipynb` (same counting method; the save step and school-code grouping were added here)
- **Reads:**
  - `data/raw/school_related/CoursesTaught18.Txt` (CDE "Courses Taught" 2017-18, 200 MB, **not in this repo**: download from CDE and place it there)
  - `data/raw/school_related/ClassCodes.xlsx` (CDE course code descriptions)
- **Writes:**
  - `data/derived/school_AP_IB_courses.pkl`

> Note: The output file is already included in `data/derived/`, so this notebook is only needed to rebuild it. Running it reproduces the shipped file exactly (all 9,404 schools, identical AP and IB counts; checked October 2026). Course data were only available through 2017-18, so these counts are used for all years (see manuscript, Section 2.1).

In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
import pandas as pd

courses_file = DATA_DIR / "school_related" / "CoursesTaught18.Txt"
if not courses_file.exists():
    raise FileNotFoundError(
        f"{courses_file} not found. Download the 2017-18 'Courses Taught' file from the "
        "California Department of Education and place it there. "
        "The output of this notebook is already in data/derived/."
    )

courses = pd.read_csv(courses_file, sep="\t", encoding="latin-1", low_memory=False)
class_codes = pd.read_excel(DATA_DIR / "school_related" / "ClassCodes.xlsx", skiprows=1)

In [ ]:
# Attach course names to course codes
courses["CourseCode"] = courses["CourseCode"].astype(str)
class_codes["Coded Value"] = class_codes["Coded Value"].astype(str)
df = courses.merge(class_codes, left_on="CourseCode", right_on="Coded Value", how="left")

# Count each distinct course once per school
df = df.drop_duplicates(subset=["SchoolName", "Name"]).copy()
df["Name"] = df["Name"].fillna("")
df["IB"] = df["Name"].str.contains("IB", case=True).astype(int)
df["AP"] = df["Name"].str.contains("AP", case=True).astype(int)

ap_ib = (df.groupby(["SchoolName", "schoolCode", "DistrictName", "CountyName"])[["IB", "AP"]]
           .sum().reset_index()
           .rename(columns={"IB": "IB_Count", "AP": "AP_Count"}))
ap_ib.sort_values("AP_Count", ascending=False).head(10)

In [ ]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
ap_ib.to_pickle(DERIVED_DIR / "school_AP_IB_courses.pkl")
ap_ib.to_csv(DERIVED_DIR / "school_AP_IB_courses.csv", index=False)
print(len(ap_ib), "schools written")